# Load TPC-H: MinIO (tpch-raw) → Iceberg (lakehouse) → Read Parquet
#
# Prasyarat: jalankan dulu upload-tpch-to-minio.sh untuk upload file .tbl ke MinIO

In [1]:
import os
from pyspark.sql import SparkSession

JARS = ','.join([
    '/home/jovyan/extra-jars/iceberg-spark-runtime-3.5_2.12-1.4.3.jar',
    '/home/jovyan/extra-jars/hadoop-aws-3.3.4.jar',
    '/home/jovyan/extra-jars/aws-java-sdk-bundle-1.12.262.jar',
])
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--jars {JARS} pyspark-shell'

spark = (
    SparkSession.builder
    .appName('tpch-load')
    .master('local[*]')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', 's3a://lakehouse/')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin123')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

print('Spark version:', spark.version)

Spark version: 3.5.0


## 1. Load .tbl → Iceberg

In [2]:
RAW_BASE = 's3a://tpch-raw'

TABLES = {
    'customer': 'c_custkey LONG, c_name STRING, c_address STRING, c_nationkey LONG, c_phone STRING, c_acctbal DECIMAL(15,2), c_mktsegment STRING, c_comment STRING, _trailing STRING',
    'orders':   'o_orderkey LONG, o_custkey LONG, o_orderstatus STRING, o_totalprice DECIMAL(15,2), o_orderdate DATE, o_orderpriority STRING, o_clerk STRING, o_shippriority INT, o_comment STRING, _trailing STRING',
    'lineitem': 'l_orderkey LONG, l_partkey LONG, l_suppkey LONG, l_linenumber INT, l_quantity DECIMAL(15,2), l_extendedprice DECIMAL(15,2), l_discount DECIMAL(15,2), l_tax DECIMAL(15,2), l_returnflag STRING, l_linestatus STRING, l_shipdate DATE, l_commitdate DATE, l_receiptdate DATE, l_shipinstruct STRING, l_shipmode STRING, l_comment STRING, _trailing STRING',
    'part':     'p_partkey LONG, p_name STRING, p_mfgr STRING, p_brand STRING, p_type STRING, p_size INT, p_container STRING, p_retailprice DECIMAL(15,2), p_comment STRING, _trailing STRING',
    'supplier': 's_suppkey LONG, s_name STRING, s_address STRING, s_nationkey LONG, s_phone STRING, s_acctbal DECIMAL(15,2), s_comment STRING, _trailing STRING',
    'partsupp': 'ps_partkey LONG, ps_suppkey LONG, ps_availqty INT, ps_supplycost DECIMAL(15,2), ps_comment STRING, _trailing STRING',
    'nation':   'n_nationkey LONG, n_name STRING, n_regionkey LONG, n_comment STRING, _trailing STRING',
    'region':   'r_regionkey LONG, r_name STRING, r_comment STRING, _trailing STRING',
}

spark.sql('CREATE NAMESPACE IF NOT EXISTS local.tpch')

for table, schema in TABLES.items():
    df = (
        spark.read
        .option('delimiter', '|')
        .option('header', 'false')
        .schema(schema)
        .csv(f'{RAW_BASE}/{table}.tbl')
        .drop('_trailing')
    )
    (
        df.writeTo(f'local.tpch.{table}')
        .tableProperty('write.format.default', 'parquet')
        .tableProperty('write.parquet.compression-codec', 'snappy')
        .using('iceberg')
        .createOrReplace()
    )
    print(f'OK {table}: {df.count():,} rows → local.tpch.{table}')

OK customer: 150,000 rows → local.tpch.customer
OK orders: 1,500,000 rows → local.tpch.orders
OK lineitem: 6,001,215 rows → local.tpch.lineitem
OK part: 200,000 rows → local.tpch.part
OK supplier: 10,000 rows → local.tpch.supplier
OK partsupp: 800,000 rows → local.tpch.partsupp
OK nation: 25 rows → local.tpch.nation
OK region: 5 rows → local.tpch.region


## 2. Baca dari Iceberg

In [3]:
spark.sql('SHOW TABLES IN local.tpch').show()

spark.sql('SELECT * FROM local.tpch.customer LIMIT 5').show(truncate=False)
spark.sql('SELECT * FROM local.tpch.orders LIMIT 5').show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     tpch|   orders|      false|
|     tpch|   nation|      false|
|     tpch| lineitem|      false|
|     tpch| customer|      false|
|     tpch|   region|      false|
|     tpch| partsupp|      false|
|     tpch| supplier|      false|
|     tpch|     part|      false|
+---------+---------+-----------+

+---------+------------------+------------------------------+-----------+---------------+---------+------------+------------------------------------------------------------------------------------------------------+
|c_custkey|c_name            |c_address                     |c_nationkey|c_phone        |c_acctbal|c_mktsegment|c_comment                                                                                             |
+---------+------------------+------------------------------+-----------+---------------+---------+------------+-----------------------------------------------

In [4]:
# Cek row count semua tabel
for table in TABLES:
    try:
        n = spark.sql(f'SELECT COUNT(*) FROM local.tpch.{table}').collect()[0][0]
        print(f'{table:12s}: {n:,} rows')
    except Exception as e:
        print(f'{table:12s}: error — {e}')

customer    : 150,000 rows
orders      : 1,500,000 rows
lineitem    : 6,001,215 rows
part        : 200,000 rows
supplier    : 10,000 rows
partsupp    : 800,000 rows
nation      : 25 rows
region      : 5 rows


## 3. Perbandingan Ukuran: CSV vs Parquet

In [7]:
from py4j.java_gateway import java_import

java_import(spark._jvm, 'org.apache.hadoop.fs.FileSystem')
java_import(spark._jvm, 'org.apache.hadoop.fs.Path')
java_import(spark._jvm, 'java.net.URI')

conf = spark._jsc.hadoopConfiguration()

def bucket_size_bytes(bucket_uri):
    fs = spark._jvm.FileSystem.get(spark._jvm.URI(bucket_uri), conf)
    path = spark._jvm.Path(bucket_uri)
    return fs.getContentSummary(path).getLength()

def fmt(b):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if b < 1024:
            return f'{b:.2f} {unit}'
        b /= 1024
    return f'{b:.2f} TB'

csv_bytes     = bucket_size_bytes('s3a://tpch-raw/')
parquet_bytes = bucket_size_bytes('s3a://lakehouse/')

print(f'CSV    (tpch-raw) : {fmt(csv_bytes):>12}  ({csv_bytes:,} bytes)')
print(f'Parquet (lakehouse): {fmt(parquet_bytes):>12}  ({parquet_bytes:,} bytes)')
print(f'Rasio kompresi    : {csv_bytes / parquet_bytes:.2f}x lebih kecil')

CSV    (tpch-raw) :      1.02 GB  (1,092,031,885 bytes)
Parquet (lakehouse):    313.89 MB  (329,136,827 bytes)
Rasio kompresi    : 3.32x lebih kecil
